# نگاشت و اعتبارسنجی جدول

کنترل ساختار جای تأیید صحت اقتصادی یا پوشش تاریخی را نمی‌گیرد.

### سلول ۱ — ورود کتابخانه‌ها

In [ ]:
import pandas as pd
from pydara import tse
from pydara.tse.exceptions import DataQualityError

### سلول ۲ — تابع نمایش مسئله‌ها

In [ ]:
def inspect_table(frame):
    report = tse.validate(frame, schema="daily_bars", strict=False)
    display(report)
    return report.attrs["pydara"]["is_valid"]

### سلول ۳ — تنظیمات نگاشت

In [ ]:
COLUMN_MAP = {"پایانی": "closing_price", "شناسه": "ins_code", "روز": "date"}

### سلول ۴ — ساخت ورودی خارجی نمونه

In [ ]:
original = tse.load_sample()
external = original.rename(columns={"closing_price": "پایانی", "ins_code": "شناسه", "date": "روز"}).copy()
external["پایانی"] = external["پایانی"].astype("string")
external["روز"] = external["روز"].dt.strftime("%Y-%m-%d")
external.attrs = {}
before = external.copy(deep=True)
normalized = tse.normalize_data(external, columns=COLUMN_MAP, calendar="gregorian")
pd.testing.assert_frame_equal(external, before)
display(tse.select_columns(normalized))
print("ساختار معتبر:", inspect_table(normalized))

### سلول ۵ — نقص داده آشکار می‌شود و حذف نمی‌شود

In [ ]:
bad = pd.concat([normalized, normalized.iloc[[0]]], ignore_index=True)
bad.loc[1, "closing_price"] = -10.0
print("ساختار معتبر:", inspect_table(bad))
assert len(bad) == len(normalized) + 1
try:
    tse.validate(bad, schema="daily_bars", strict=True)
except DataQualityError as error:
    print(type(error).__name__)
    display(error.report)

### سلول ۶ — عدد دقیق و ستون‌های انتخابی

In [ ]:
exact = tse.normalize_data(normalized, numeric="exact")
print(type(exact.loc[0, "closing_price"]).__name__)
display(tse.select_columns(exact, ["date", "ins_code", "closing_price"]))